We will use Logistic Regression as our model, because it is suitable for the features we have prepared and for the resulting sparse array from One-Hot Encoding.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from scipy import sparse
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [2]:
artifacts_dir = Path("../artifacts")

print("Artifacts directory:", artifacts_dir)

Artifacts directory: ..\artifacts


2.Reading Feature Tables and Labels

In [3]:
X_train = sparse.load_npz(artifacts_dir / "X_train.npz")

X_validation = sparse.load_npz(artifacts_dir / "X_validation.npz")

X_test = sparse.load_npz(artifacts_dir / "X_test.npz")

y_train = pd.read_csv(artifacts_dir / "y_train.csv")["late_delivery"]

y_validation = pd.read_csv(artifacts_dir / "y_validation.csv")["late_delivery"]

y_test = pd.read_csv(artifacts_dir / "y_test.csv")["late_delivery"]

3.Checking the sizes

In [4]:
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

X_train: (69608, 3818)
X_validation: (14916, 3818)
X_test: (14917, 3818)
y_train: (69608,)
y_validation: (14916,)
y_test: (14917,)


4.Baseline

In [5]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.91,0.09]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,3818
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [6]:
baseline_pred = baseline.predict(X_validation)

In [7]:
baseline_accuracy = accuracy_score(y_validation, baseline_pred)

baseline_precision = precision_score(y_validation, baseline_pred, zero_division=0)

baseline_recall = recall_score(y_validation, baseline_pred, zero_division=0)

baseline_f1 = f1_score(y_validation, baseline_pred, zero_division=0)

print("Baseline Results")
print("----------------")
print("Accuracy :", baseline_accuracy)
print("Precision:", baseline_precision)
print("Recall   :", baseline_recall)
print("F1 Score :", baseline_f1)

Baseline Results
----------------
Accuracy : 0.9473049074818987
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0


5.Model Training

In [8]:
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

model.fit(X_train, y_train)

c:\Users\saefa\OneDrive\Desktop\olist project\olist project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to u

6.Model evaluation on Validation

In [9]:
validation_pred = model.predict(X_validation)

validation_prob = model.predict_proba(X_validation)[:, 1]

In [10]:
validation_accuracy = accuracy_score(y_validation, validation_pred)

validation_precision = precision_score(y_validation, validation_pred, zero_division=0)

validation_recall = recall_score(y_validation, validation_pred, zero_division=0)

validation_f1 = f1_score(y_validation, validation_pred, zero_division=0)

validation_roc_auc = roc_auc_score(y_validation, validation_prob)

print("Validation Results")
print("-------------------")
print("Accuracy :", validation_accuracy)
print("Precision:", validation_precision)
print("Recall   :", validation_recall)
print("F1 Score :", validation_f1)
print("ROC-AUC  :", validation_roc_auc)

Validation Results
-------------------
Accuracy : 0.5909761330115313
Precision: 0.08919462049775854
Recall   : 0.7340966921119593
F1 Score : 0.15906271536871122
ROC-AUC  : 0.7078836737744211


7.Adjusting the hyperparameter

In [11]:
C_values = [0.01, 0.1, 1, 10]

tuning_results = []

for c in C_values:
    tuned_model = LogisticRegression(
        C=c, max_iter=1000, class_weight="balanced", random_state=42
    )

    tuned_model.fit(X_train, y_train)

    val_pred = tuned_model.predict(X_validation)

    val_prob = tuned_model.predict_proba(X_validation)[:, 1]

    result = {
        "C": c,
        "accuracy": accuracy_score(y_validation, val_pred),
        "precision": precision_score(y_validation, val_pred, zero_division=0),
        "recall": recall_score(y_validation, val_pred, zero_division=0),
        "f1": f1_score(y_validation, val_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_validation, val_prob),
    }

    tuning_results.append(result)

tuning_results_df = pd.DataFrame(tuning_results)

print(tuning_results_df)

c:\Users\saefa\OneDrive\Desktop\olist project\olist project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\saefa\OneDrive\Desktop\olist project\olist project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as s

       C  accuracy  precision    recall        f1   roc_auc
0   0.01  0.637168   0.093783  0.679389  0.164815  0.712387
1   0.10  0.641593   0.093148  0.664122  0.163380  0.703909
2   1.00  0.590976   0.089195  0.734097  0.159063  0.707884
3  10.00  0.647493   0.094781  0.665394  0.165926  0.709922


c:\Users\saefa\OneDrive\Desktop\olist project\olist project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


8.Choosing the best hyperparameter

In [12]:
best_result = tuning_results_df.loc[tuning_results_df["f1"].idxmax()]

best_C = best_result["C"]

print("Best C:", best_C)
print()
print(best_result)

Best C: 10.0

C            10.000000
accuracy      0.647493
precision     0.094781
recall        0.665394
f1            0.165926
roc_auc       0.709922
Name: 3, dtype: float64


9.Best Model Training

In [13]:
best_model = LogisticRegression(
    C=best_C, max_iter=1000, class_weight="balanced", random_state=42
)

best_model.fit(X_train, y_train)

c:\Users\saefa\OneDrive\Desktop\olist project\olist project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",np.float64(10.0)
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'A

In [14]:
joblib.dump(best_model, artifacts_dir / "trained_model.pkl")

print("Best model saved successfully.")

Best model saved successfully.


10.Confirmation of the validation result for the best model

In [15]:
best_validation_pred = best_model.predict(X_validation)

best_validation_prob = best_model.predict_proba(X_validation)[:, 1]

best_validation_accuracy = accuracy_score(y_validation, best_validation_pred)

best_validation_precision = precision_score(
    y_validation, best_validation_pred, zero_division=0
)

best_validation_recall = recall_score(
    y_validation, best_validation_pred, zero_division=0
)

best_validation_f1 = f1_score(y_validation, best_validation_pred, zero_division=0)

best_validation_roc_auc = roc_auc_score(y_validation, best_validation_prob)

print("Best Model - Validation Results")
print("--------------------------------")
print("Accuracy :", best_validation_accuracy)
print("Precision:", best_validation_precision)
print("Recall   :", best_validation_recall)
print("F1 Score :", best_validation_f1)
print("ROC-AUC  :", best_validation_roc_auc)

Best Model - Validation Results
--------------------------------
Accuracy : 0.6474926253687315
Precision: 0.09478071765132294
Recall   : 0.6653944020356234
F1 Score : 0.16592639593908629
ROC-AUC  : 0.709922448582681


11.Test Assessment

In [16]:
test_pred = best_model.predict(X_test)

test_prob = best_model.predict_proba(X_test)[:, 1]

In [17]:
test_accuracy = accuracy_score(y_test, test_pred)

test_precision = precision_score(y_test, test_pred, zero_division=0)

test_recall = recall_score(y_test, test_pred, zero_division=0)

test_f1 = f1_score(y_test, test_pred, zero_division=0)

test_roc_auc = roc_auc_score(y_test, test_prob)

print("FINAL TEST RESULTS")
print("-------------------")
print("Accuracy :", test_accuracy)
print("Precision:", test_precision)
print("Recall   :", test_recall)
print("F1 Score :", test_f1)
print("ROC-AUC  :", test_roc_auc)

FINAL TEST RESULTS
-------------------
Accuracy : 0.4571294496212375
Precision: 0.08225108225108226
Recall   : 0.7345872518286312
F1 Score : 0.14793771043771045
ROC-AUC  : 0.5926726009227739


Confusion Matrix

In [18]:
cm = confusion_matrix(y_test, test_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[6116 7844]
 [ 254  703]]


Results Summary

In [19]:
results_summary = f"""
Model Evaluation Summary

Baseline:
Accuracy  : {baseline_accuracy:.4f}
Precision : {baseline_precision:.4f}
Recall    : {baseline_recall:.4f}
F1 Score  : {baseline_f1:.4f}

Best Hyperparameter:
C : {best_C}

Validation:
Accuracy  : {best_validation_accuracy:.4f}
Precision : {best_validation_precision:.4f}
Recall    : {best_validation_recall:.4f}
F1 Score  : {best_validation_f1:.4f}
ROC-AUC   : {best_validation_roc_auc:.4f}

Final Test:
Accuracy  : {test_accuracy:.4f}
Precision : {test_precision:.4f}
Recall    : {test_recall:.4f}
F1 Score  : {test_f1:.4f}
ROC-AUC   : {test_roc_auc:.4f}

Confusion Matrix:
{cm}
"""

print(results_summary)


Model Evaluation Summary

Baseline:
Accuracy  : 0.9473
Precision : 0.0000
Recall    : 0.0000
F1 Score  : 0.0000

Best Hyperparameter:
C : 10.0

Validation:
Accuracy  : 0.6475
Precision : 0.0948
Recall    : 0.6654
F1 Score  : 0.1659
ROC-AUC   : 0.7099

Final Test:
Accuracy  : 0.4571
Precision : 0.0823
Recall    : 0.7346
F1 Score  : 0.1479
ROC-AUC   : 0.5927

Confusion Matrix:
[[6116 7844]
 [ 254  703]]



In [20]:
results_path = artifacts_dir / "results_summary.txt"

with open(results_path, "w", encoding="utf-8") as f:
    f.write(results_summary)

print("Results summary saved to:", results_path)

Results summary saved to: ..\artifacts\results_summary.txt


In [21]:
print("Final Notebook 6 artifacts:")

for path in sorted(artifacts_dir.glob("*")):
    print(path)

Final Notebook 6 artifacts:
..\artifacts\categorical_imputer.pkl
..\artifacts\eda_charts
..\artifacts\eda_findings_summary.txt
..\artifacts\feature_list.txt
..\artifacts\labeled_table.csv
..\artifacts\ml_table.csv
..\artifacts\numeric_imputer.pkl
..\artifacts\onehot_encoder.pkl
..\artifacts\results_summary.txt
..\artifacts\test.csv
..\artifacts\train.csv
..\artifacts\trained_model.pkl
..\artifacts\validation.csv
..\artifacts\X_test.npz
..\artifacts\X_train.npz
..\artifacts\X_validation.npz
..\artifacts\y_test.csv
..\artifacts\y_train.csv
..\artifacts\y_validation.csv


Findings : 


The Logistic Regression model improved substantially over the majority-class baseline in detecting late deliveries, achieving a recall of 73.46% on the test set. However, precision was very low at 8.23%, resulting in a low F1 score of 14.79%. The ROC-AUC of 0.593 on the test set also indicates limited generalization compared with the validation result.

In [22]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")

mlflow.set_experiment("olist-late-delivery")

c:\Users\saefa\OneDrive\Desktop\olist project\olist project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/23 23:00:44 INFO mlflow.tracking.fluent: Experiment with name 'olist-late-delivery' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1790193644526, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790193644526, lifecycle_stage='active', name='olist-late-delivery', tags={}, trace_location=None, workspace='default'>

In [23]:
with mlflow.start_run() as run:
    mlflow.log_param("model", "LogisticRegression")

    mlflow.log_param("C", best_C)

    mlflow.log_param("class_weight", "balanced")

    mlflow.log_metric("validation_accuracy", best_validation_accuracy)

    mlflow.log_metric("validation_precision", best_validation_precision)

    mlflow.log_metric("validation_recall", best_validation_recall)

    mlflow.log_metric("validation_f1", best_validation_f1)

    mlflow.log_metric("validation_roc_auc", best_validation_roc_auc)

    mlflow.log_metric("test_accuracy", test_accuracy)

    mlflow.log_metric("test_precision", test_precision)

    mlflow.log_metric("test_recall", test_recall)

    mlflow.log_metric("test_f1", test_f1)

    mlflow.log_metric("test_roc_auc", test_roc_auc)

    mlflow.log_artifact(
        "../artifacts/feature_list.txt", artifact_path="inference_artifacts"
    )

    mlflow.log_artifact(
        "../artifacts/numeric_imputer.pkl", artifact_path="inference_artifacts"
    )

    mlflow.log_artifact(
        "../artifacts/categorical_imputer.pkl", artifact_path="inference_artifacts"
    )

    mlflow.log_artifact(
        "../artifacts/onehot_encoder.pkl", artifact_path="inference_artifacts"
    )

    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="model",
        registered_model_name=("olist-late-delivery-model"),
    )

    print("MLflow run:", run.info.run_id)

Successfully registered model 'olist-late-delivery-model'.
2026/09/23 23:01:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: olist-late-delivery-model, version 1
Created version '1' of model 'olist-late-delivery-model'.


MLflow run: 5ee74c0f4f2644498dd3437ad5050cb6
🏃 View run masked-horse-257 at: http://localhost:5000/#/experiments/1/runs/5ee74c0f4f2644498dd3437ad5050cb6
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [24]:
from mlflow import MlflowClient

client = MlflowClient(tracking_uri="http://localhost:5000")

versions = client.search_model_versions("name='olist-late-delivery-model'")

latest_version = max(versions, key=lambda item: int(item.version))

client.transition_model_version_stage(
    name="olist-late-delivery-model",
    version=latest_version.version,
    stage="Production",
    archive_existing_versions=True,
)

print(
    "Registered version:",
    latest_version.version,
)

print(
    "Stage:",
    "Production",
)

Registered version: 1
Stage: Production


C:\Users\saefa\AppData\Local\Temp\ipykernel_21400\524266551.py:16: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
